In [1]:
import pandas as pd
import sqlite3

# Carrega os dados tratados
df = pd.read_csv("../data/processed/cobertura_vacinal_por_uf.csv")

# Cria (ou conecta a) um banco SQLite
conn = sqlite3.connect("../data/processed/cobertura_vacinal.db")

# Envia o DataFrame como uma tabela SQL
df.to_sql("cobertura_vacinal", conn, if_exists="replace", index=False)

print("Banco de dados criado com sucesso!")

Banco de dados criado com sucesso!


In [2]:
query_teste = "SELECT * FROM cobertura_vacinal LIMIT 5;"
pd.read_sql(query_teste, conn)

,codigo_uf,uf,ano,cobertura,vacina
0,11,Rondônia,2013,108.15,BCG
1,12,Acre,2013,106.16,BCG
2,13,Amazonas,2013,116.92,BCG
3,14,Roraima,2013,94.17,BCG
4,15,Pará,2013,117.68,BCG


In [3]:
query1 = """
SELECT
    uf,
    vacina,
    ano,
    cobertura,
    LAG(cobertura) OVER (PARTITION BY uf, vacina ORDER BY ano) AS cobertura_ano_anterior,
    ROUND(cobertura - LAG(cobertura) OVER (PARTITION BY uf, vacina ORDER BY ano), 2) AS variacao_absoluta
FROM cobertura_vacinal
WHERE uf = 'Roraima'
ORDER BY vacina, ano;
"""

pd.read_sql(query1, conn)

,uf,vacina,ano,cobertura,cobertura_ano_anterior,variacao_absoluta
0,Roraima,BCG,2013,94.17,NaN,NaN
1,Roraima,BCG,2014,103.97,94.17,9.80
2,Roraima,BCG,2015,110.55,103.97,6.58
3,Roraima,BCG,2016,107.95,110.55,-2.60
4,Roraima,BCG,2017,116.74,107.95,8.79
5,Roraima,BCG,2018,135.55,116.74,18.81
6,Roraima,BCG,2019,115.88,135.55,-19.67
7,Roraima,BCG,2020,99.76,115.88,-16.12
8,Roraima,BCG,2021,77.13,99.76,-22.63
9,Roraima,BCG,2022,85.17,77.13,8.04


In [4]:
query2 = """
WITH variacao_por_estado AS (
    SELECT
        uf,
        AVG(CASE WHEN ano = 2019 THEN cobertura END) AS cobertura_2019,
        AVG(CASE WHEN ano = 2022 THEN cobertura END) AS cobertura_2022
    FROM cobertura_vacinal
    WHERE ano IN (2019, 2022)
    GROUP BY uf
)
SELECT
    uf,
    ROUND(cobertura_2019, 2) AS cobertura_2019,
    ROUND(cobertura_2022, 2) AS cobertura_2022,
    ROUND(cobertura_2022 - cobertura_2019, 2) AS variacao,
    RANK() OVER (ORDER BY (cobertura_2022 - cobertura_2019) ASC) AS ranking_queda
FROM variacao_por_estado
ORDER BY ranking_queda
LIMIT 10;
"""

pd.read_sql(query2, conn)

,uf,cobertura_2019,cobertura_2022,variacao,ranking_queda
0,Roraima,85.25,68.22,-17.03,1
1,Paraíba,93.80,79.44,-14.36,2
2,Amapá,78.47,64.18,-14.29,3
3,Mato Grosso do Sul,99.95,87.03,-12.92,4
4,Rio de Janeiro,76.25,65.03,-11.23,5
5,Acre,84.88,73.81,-11.07,6
6,Espírito Santo,84.72,74.95,-9.77,7
7,Rondônia,95.72,89.24,-6.48,8
8,São Paulo,83.62,78.62,-5.00,9
9,Pernambuco,88.10,83.19,-4.91,10


In [5]:
query3 = """
SELECT
    vacina,
    ano,
    ROUND(AVG(cobertura), 2) AS cobertura_media_nacional,
    ROUND(AVG(AVG(cobertura)) OVER (
        PARTITION BY vacina 
        ORDER BY ano 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS media_movel_3_anos
FROM cobertura_vacinal
GROUP BY vacina, ano
ORDER BY vacina, ano;
"""

pd.read_sql(query3, conn)

,vacina,ano,cobertura_media_nacional,media_movel_3_anos
0,BCG,2013,108.16,108.16
1,BCG,2014,108.89,108.53
2,BCG,2015,106.03,107.69
3,BCG,2016,98.57,104.50
4,BCG,2017,97.89,100.83
5,BCG,2018,102.26,99.57
6,BCG,2019,90.94,97.03
7,BCG,2020,79.92,91.04
8,BCG,2021,78.89,83.25
9,BCG,2022,93.89,84.23
